# 📖 Notebook 2: Basic Workflows and Activities

Now that we understand **why** Temporal exists, let's write our first workflow!
We'll connect to a local Temporal server, define activities and workflows, and run them.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Connect to a Temporal server from Python
- Define **activities** (functions that do real work)
- Define **workflows** (functions that orchestrate activities)
- Run a workflow and get its result
- Configure retry policies for activities
- Use the Temporal Web UI to inspect workflow history

## 🛠️ Setup

Make sure Temporal is running first:

```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
```

Wait ~30 seconds for Temporal to initialize, then verify:

```bash
docker compose ps   # All containers should be 'running' or 'healthy'
```

### Temporal Web UI

Open http://localhost:8080 in your browser — this is where you can inspect workflows.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
# Step 1: Connect to the Temporal server

from temporalio.client import Client

client = await Client.connect("localhost:7233")
print("✅ Connected to Temporal server at localhost:7233")
print("   Open http://localhost:8080 to see the Temporal Web UI")

## 📦 What is an Activity?

An **activity** is a function that does real work — the things that have **side effects**:
- Calling an external API (Stripe, FedEx, etc.)
- Reading from or writing to a database
- Sending an email
- Processing a file

Activities are:
- **Retryable** — Temporal automatically retries them if they fail
- **Not deterministic** — they can use `random`, `datetime.now()`, network calls, etc.
- **Isolated** — each activity runs independently

You create an activity by decorating a function with `@activity.defn`:

In [ ]:
# Step 2: Define some activities

import asyncio
from temporalio import activity


@activity.defn
async def say_hello(name: str) -> str:
    """A simple activity that greets someone."""
    activity.logger.info(f"Saying hello to {name}")
    await asyncio.sleep(0.5)  # simulate some work
    return f"Hello, {name}!"


@activity.defn
async def format_greeting(greeting: str) -> str:
    """Add decoration to a greeting."""
    activity.logger.info(f"Formatting: {greeting}")
    await asyncio.sleep(0.3)
    return f"🎉 {greeting} Welcome to Temporal! 🎉"


print("✅ Defined 2 activities: say_hello, format_greeting")
print()
print("Key points:")
print("  • @activity.defn marks a function as a Temporal activity")
print("  • Activities can be async (recommended) or sync")
print("  • Use activity.logger instead of print() for proper logging")
print("  • Activities take simple inputs and return simple outputs")

## 🔄 What is a Workflow?

A **workflow** is a class that **orchestrates** activities. It defines the order,
handles errors, and manages the overall process.

Workflows have strict rules:
1. **Must be deterministic** — no `random()`, no `datetime.now()`, no network calls
2. **No side effects** — all real work goes in activities
3. **Can be replayed** — Temporal re-runs your workflow code from event history on recovery

Think of it this way:
- A **workflow** is the recipe: "First mix flour, then add eggs, then bake at 350°"
- An **activity** is the actual cooking: mixing, cracking eggs, using the oven

The recipe never changes, but each step might need to be retried if something goes wrong.

In [ ]:
# Step 3: Define a workflow that uses our activities

from datetime import timedelta
from temporalio import workflow

# This import is required — workflow code runs in a sandbox, so activities
# must be imported through this special context manager.
with workflow.unsafe.imports_passed_through():
    # We already defined these above, but in a real project they'd be
    # imported from a separate file
    pass


@workflow.defn
class GreetingWorkflow:
    """A simple workflow that greets someone with style."""

    @workflow.run
    async def run(self, name: str) -> str:
        # Step 1: Generate a greeting
        greeting = await workflow.execute_activity(
            say_hello,
            name,
            start_to_close_timeout=timedelta(seconds=10),
        )

        # Step 2: Format it nicely
        result = await workflow.execute_activity(
            format_greeting,
            greeting,
            start_to_close_timeout=timedelta(seconds=10),
        )

        return result


print("✅ Defined GreetingWorkflow")
print()
print("Key points:")
print("  • @workflow.defn marks a class as a Temporal workflow")
print("  • @workflow.run marks the entry point method")
print("  • workflow.execute_activity() calls an activity and waits for its result")
print("  • start_to_close_timeout = max time an activity can run")

## ▶️ Running Your First Workflow

To run a workflow, we need:
1. A **client** to tell Temporal "start this workflow" (we already have this)
2. A **worker** to actually execute the workflow and activity code

We'll start a worker right here in the notebook, then execute the workflow.

```
┌────────────┐       ┌──────────────┐       ┌───────────────┐
│  This cell  │──────▶│  Temporal     │──────▶│  Worker       │
│  (client)   │       │  Server      │       │  (also here)  │
│             │       │              │       │               │
│ "Run this   │       │ "Scheduling  │       │ "Running      │
│  workflow"  │       │  tasks..."   │       │  activities"  │
└────────────┘       └──────────────┘       └───────────────┘
```

In [ ]:
# Step 4: Run the workflow!
#
# We use `async with Worker(...)` to start a worker that stays alive
# while we execute the workflow. When the workflow finishes, the worker
# stops too.

import uuid
from temporalio.worker import Worker

TASK_QUEUE = "basics-task-queue"


async def run_greeting(name: str) -> str:
    """Start a worker and execute GreetingWorkflow."""
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[GreetingWorkflow],
        activities=[say_hello, format_greeting],
    ):
        # The worker is now running and polling for tasks.
        # Let's start a workflow execution:
        result = await client.execute_workflow(
            GreetingWorkflow.run,
            name,
            id=f"greeting-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )
        return result


result = await run_greeting("World")
print(f"📬 Workflow result: {result}")
print()
print("🎉 You just ran your first Temporal workflow!")
print("   Check http://localhost:8080 to see it in the Temporal UI.")

## 🔍 What Just Happened?

Behind the scenes, Temporal recorded this event history:

```
1. WorkflowExecutionStarted   — input: "World"
2. WorkflowTaskScheduled       — Temporal asks worker to run workflow code
3. WorkflowTaskCompleted       — Worker ran workflow, decided to call say_hello
4. ActivityTaskScheduled       — say_hello("World")
5. ActivityTaskCompleted       — say_hello returned "Hello, World!"
6. ActivityTaskScheduled       — format_greeting("Hello, World!")
7. ActivityTaskCompleted       — format_greeting returned the final string
8. WorkflowExecutionCompleted  — workflow returned the result
```

You can see this exact history in the Temporal UI at http://localhost:8080.

**This event history is the key to durable execution.** If the worker crashed after event 5,
Temporal would replay events 1-5, reconstruct the state, and continue from event 6.

## 🔁 Retry Policies

One of Temporal's most powerful features is **automatic retries**. When an activity fails
(throws an exception), Temporal can automatically retry it with configurable backoff.

The key settings:

| Setting | What It Does | Example |
|---------|-------------|----------|
| `initial_interval` | How long to wait before the first retry | 1 second |
| `backoff_coefficient` | Multiply wait time by this after each retry | 2.0 (1s → 2s → 4s → 8s) |
| `maximum_interval` | Cap on how long to wait between retries | 30 seconds |
| `maximum_attempts` | Give up after this many total tries | 5 |

Let's see retries in action with a "flaky" activity that fails sometimes:

In [ ]:
# Define a flaky activity that fails 60% of the time

import random
from temporalio.common import RetryPolicy

attempt_counter = 0  # track attempts across retries


@activity.defn
async def flaky_api_call(order_id: str) -> dict:
    """Simulates calling an unreliable external API."""
    global attempt_counter
    attempt_counter += 1
    current = attempt_counter

    activity.logger.info(f"Attempt {current} for order {order_id}")
    await asyncio.sleep(0.3)

    # Fail on the first 2 attempts, succeed on the 3rd
    if current <= 2:
        raise RuntimeError(f"API timeout on attempt {current}!")

    return {"order_id": order_id, "status": "confirmed", "attempt": current}


@workflow.defn
class RetryDemoWorkflow:
    """Shows how Temporal automatically retries failed activities."""

    @workflow.run
    async def run(self, order_id: str) -> dict:
        result = await workflow.execute_activity(
            flaky_api_call,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
            retry_policy=RetryPolicy(
                initial_interval=timedelta(seconds=1),
                backoff_coefficient=2.0,
                maximum_interval=timedelta(seconds=10),
                maximum_attempts=5,
            ),
        )
        return result


print("✅ Defined flaky_api_call activity and RetryDemoWorkflow")
print("   The activity fails on attempts 1-2, succeeds on attempt 3")

In [ ]:
# Run it and watch the retries

attempt_counter = 0  # reset counter


async def run_retry_demo():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[RetryDemoWorkflow],
        activities=[flaky_api_call],
    ):
        result = await client.execute_workflow(
            RetryDemoWorkflow.run,
            "ORD-001",
            id=f"retry-demo-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )
        return result


result = await run_retry_demo()
print(f"📬 Result: {result}")
print(f"\n✅ Succeeded on attempt {result['attempt']}!")
print("   Temporal automatically retried the activity after each failure.")
print("   You didn't have to write ANY retry logic — Temporal handled it all.")

## 🚫 Non-Retryable Errors

Some errors should **not** be retried. For example:
- `InvalidCreditCard` — retrying won't help
- `OrderNotFound` — the data doesn't exist
- `InsufficientFunds` — the customer can't pay

You can tell Temporal which error types are non-retryable:

In [ ]:
from temporalio import activity, workflow
from temporalio.common import RetryPolicy
from temporalio.exceptions import ApplicationError


@activity.defn
async def validate_payment(card_number: str) -> dict:
    """Validates a credit card — some errors are non-retryable."""
    activity.logger.info(f"Validating card ending in ...{card_number[-4:]}")
    await asyncio.sleep(0.3)

    if card_number.startswith("0000"):
        # This is a PERMANENT failure — don't retry!
        raise ApplicationError(
            "Invalid credit card number",
            type="InvalidCardError",  # Temporal uses this to match non_retryable_error_types
            non_retryable=True,
        )

    return {"card": f"...{card_number[-4:]}", "status": "valid"}


@workflow.defn
class PaymentValidationWorkflow:
    @workflow.run
    async def run(self, card_number: str) -> dict:
        try:
            result = await workflow.execute_activity(
                validate_payment,
                card_number,
                start_to_close_timeout=timedelta(seconds=10),
                retry_policy=RetryPolicy(maximum_attempts=5),
            )
            return {"status": "valid", "result": result}
        except Exception as e:
            return {"status": "invalid", "error": str(e)}


async def run_payment_validation(card: str):
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentValidationWorkflow],
        activities=[validate_payment],
    ):
        return await client.execute_workflow(
            PaymentValidationWorkflow.run,
            card,
            id=f"payment-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


# Test with a valid card
print("Test 1: Valid card")
r1 = await run_payment_validation("4242424242424242")
print(f"  Result: {r1}")

# Test with an invalid card — Temporal will NOT retry this
print("\nTest 2: Invalid card (starts with 0000)")
r2 = await run_payment_validation("0000111122223333")
print(f"  Result: {r2}")
print("\n💡 The invalid card failed immediately — Temporal did NOT retry it.")
print("   This saves time and avoids hammering the payment API with hopeless requests.")

## 🔎 Workflow IDs and Idempotency

Every workflow execution has a **unique ID**. If you try to start a workflow with an ID
that already exists, Temporal will reject it. This prevents duplicate processing.

Best practice: use a **business identifier** as the workflow ID:

```python
# ✅ Good — natural deduplication
id = f"order-{order_id}"  

# ❌ Bad — allows duplicates
id = f"order-{uuid.uuid4()}"
```

If a customer clicks "Place Order" twice, the second attempt gets rejected because
`order-ORD-001` already exists.

In [ ]:
# Demonstrate workflow ID uniqueness

from temporalio.client import WorkflowAlreadyStartedError

fixed_id = f"idempotency-demo-{uuid.uuid4()}"


async def demo_idempotency():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[GreetingWorkflow],
        activities=[say_hello, format_greeting],
    ):
        # First execution — succeeds
        print(f"Attempt 1: Starting workflow '{fixed_id}'")
        r1 = await client.execute_workflow(
            GreetingWorkflow.run,
            "Alice",
            id=fixed_id,
            task_queue=TASK_QUEUE,
        )
        print(f"  ✅ Result: {r1}")

        # Second execution with same ID — rejected!
        print(f"\nAttempt 2: Starting workflow '{fixed_id}' AGAIN")
        try:
            await client.execute_workflow(
                GreetingWorkflow.run,
                "Alice",
                id=fixed_id,
                task_queue=TASK_QUEUE,
            )
        except WorkflowAlreadyStartedError as e:
            print(f"  ❌ Rejected: {e}")
            print("\n💡 Temporal prevented duplicate execution!")
            print("   Use business IDs (like order-123) to get natural deduplication.")


await demo_idempotency()

## 🌐 Exploring the Temporal Web UI

Open http://localhost:8080 in your browser. You should see:

1. **Workflow list** — all the workflows we just ran
2. Click on any workflow to see its **event history**:
   - Each activity start and completion
   - Input and output for each activity
   - Timing information
   - Any retry attempts

```
┌────────────────────────────────────────────────────────────────┐
│  Temporal Web UI (http://localhost:8080)                       │
│                                                                │
│  Workflows                                                     │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │ greeting-abc123  │ GreetingWorkflow │ Completed │ 1.2s   │  │
│  │ retry-demo-xyz   │ RetryDemoWf      │ Completed │ 3.5s   │  │
│  │ payment-def456   │ PaymentValidWf   │ Completed │ 0.8s   │  │
│  └──────────────────────────────────────────────────────────┘  │
│                                                                │
│  Click any workflow → Event History → see every step!          │
└────────────────────────────────────────────────────────────────┘
```

This visibility is **free** — you didn't write any logging or dashboard code.

## 📐 Workflow vs Activity: Quick Reference

| | Workflow | Activity |
|---|---------|----------|
| **Purpose** | Orchestrate the process | Do the actual work |
| **Decorator** | `@workflow.defn` | `@activity.defn` |
| **Deterministic?** | ✅ YES — must always produce the same result | ❌ No — can use random, time, I/O |
| **Side effects?** | ❌ No — no API calls, no DB writes | ✅ Yes — this is where they go |
| **Retryable?** | Not directly — workflows are replayed | ✅ Yes — configurable retry policy |
| **Analogy** | The recipe | One cooking step |

### ⚠️ Common Mistake

**Don't put business logic directly in the workflow:**

```python
# ❌ BAD — calling an API directly in workflow code
@workflow.run
async def run(self, order_id):
    response = await httpx.get(f"https://api.stripe.com/...")  # NOT ALLOWED

# ✅ GOOD — put the API call in an activity
@workflow.run
async def run(self, order_id):
    result = await workflow.execute_activity(charge_payment, order_id, ...)
```

Workflow code must be deterministic because Temporal **replays** it during recovery.

## 📚 Summary

### What We Learned

1. **Activities** do real work (API calls, DB writes) — decorated with `@activity.defn`
2. **Workflows** orchestrate activities — decorated with `@workflow.defn`
3. **Workers** execute your code — created with `Worker(client, task_queue=..., ...)`
4. **Retry policies** handle transient failures automatically
5. **Non-retryable errors** prevent wasting time on permanent failures
6. **Workflow IDs** prevent duplicate processing
7. **Temporal UI** gives you full visibility for free

### Next Up

In **Notebook 3**, we'll tackle the **Saga Pattern** — how to undo completed steps
when a multi-step process fails halfway through.